# 3일차 1교시 — DDPG 소개

**PyTorch로 배우는 강화학습 · 3일차 Advanced Actor-Critic Methods · 2026-07-29 (수)**

이애본 (Ph.D Aebon) · DreamIT Biz · https://pytorch26.dreamitbiz.com

---

## 🎯 학습목표

- 연속 행동 공간에서 DQN이 동작하지 않는 이유를 이해한다
- 결정적 정책 경사와 DDPG의 4개 네트워크 구조를 설명할 수 있다

---

# ⚡ 실행 방법 두 가지 — 편한 쪽을 고르세요

### 방법 ① 통째로 한 번에
바로 아래 **[통째로 실행]** 셀 **하나만** 실행하면 끝까지 돕니다.
결과부터 보고 싶으신 분께 권합니다.

### 방법 ② 단계별로 하나씩
그 아래 **[단계별]** 부분을 위에서부터 `Shift + Enter` 로 하나씩 실행하세요.
모두 **6칸**입니다. 한 칸 돌리고 결과 보고 넘어가면 됩니다.

> **이 교시는 혼자 돌아갑니다.** 앞 교시를 먼저 실행하지 않아도 됩니다.
> (앞 교시에서 만든 것을 이 노트북 안에 다시 넣어 뒀습니다 — 사이트의 *이 교시 전체 코드* 와 같은 판입니다.)
> 설치할 것도 없습니다 — 코랩에 다 들어 있습니다.

---

# ① 통째로 한 번에 실행

GitHub 에서 원본을 받아 그대로 돌립니다. 원본이 고쳐지면 자동으로 최신을 받습니다.

In [ ]:
!curl -sL https://raw.githubusercontent.com/aebonlee/pytorch26-lab/main/day3/standalone/01_ddpg_networks.py -o 01_ddpg_networks.py
!python 01_ddpg_networks.py

---

# ② 단계별로 하나씩 실행

이 교시 내용이 **6칸**입니다.
위에서부터 `Shift + Enter`.

> ①을 이미 돌리셨어도 상관없습니다. 처음부터 다시 하는 것과 같습니다.

### 1 / 6 칸

In [ ]:
# ============================================================
# 3일차 1교시 — DDPG 소개
# 복사해서 그대로 실행하면 됩니다. 고칠 것 없습니다.
# ------------------------------------------------------------
# 이 교시 코드는 앞 교시의 변수·클래스를 이어 씁니다.
# 그래서 이 블록에는 **여기까지 필요한 코드가 전부** 들어 있습니다.
# (수업용 코드만 따로 복사하면 NameError 가 납니다 — 그건 정상입니다.)
# ============================================================

# ── 2일차 1교시에서 이어받음 ──
import random                            # 여러 개 중에서 아무거나 뽑을 때 쓰는 도구
from collections import deque             # 앞뒤로 넣고 빼기 쉬운 '줄서기 상자'
import numpy as np                        # 숫자 계산 도구 (파이썬의 계산기)
import torch                              # 파이토치 — 신경망을 다루는 도구

### 2 / 6 칸

In [ ]:
class ReplayBuffer:
    """
    경험 재현 버퍼 — 게임하면서 겪은 일을 적어 두는 '일기장'입니다.

    왜 필요할까요?
      방금 겪은 일만 보고 배우면 비슷한 것만 연달아 보게 됩니다.
      (왼쪽으로 갔다 -> 또 왼쪽 -> 또 왼쪽 ...)
      그러면 신경망이 방금 본 것에만 맞추고 예전 것을 잊어버립니다.
      그래서 일기장에 잔뜩 적어 두고, 배울 때는 여기저기서 섞어서 꺼냅니다.

    이 상자는 3일 내내 씁니다 — 오늘 DQN, 내일 DDPG와 SAC.
    """

    def __init__(self, capacity=100_000, action_dtype=torch.int64):
        # capacity = 일기장에 몇 줄까지 적어 둘지 (10만 줄)
        #            넘치면 가장 오래된 것부터 자동으로 지워집니다.
        #
        # action_dtype = 행동을 어떤 숫자로 적을지
        #   오늘 DQN  : 행동이 "0번? 1번?" 이라서 정수(int64)
        #   내일 DDPG : 행동이 "힘을 1.37만큼" 이라서 실수(float32)
        #   -> 내일은 ReplayBuffer(100_000, action_dtype=torch.float32) 로 씁니다.
        #      정수로 두면 1.37 이 1 로 잘려서 학습이 통째로 망가집니다.
        self.buffer = deque(maxlen=capacity)    # 실제 일기장 (꽉 차면 앞쪽부터 밀려남)
        self.action_dtype = action_dtype        # 행동을 적을 숫자 종류를 기억해 둔다

    def push(self, s, a, r, s_next, done):
        # 일기 한 줄 적기:
        #   "이 상황(s)에서 이 행동(a)을 했더니 점수(r)를 받고
        #    저 상황(s_next)이 됐다. 그리고 판이 끝났나?(done)"
        self.buffer.append((s, a, r, s_next, done))   # 다섯 개를 한 묶음으로 저장

    def sample(self, batch_size):
        # 일기장에서 batch_size 줄을 무작위로 뽑아 온다 (= 섞어서 꺼내기)
        batch = random.sample(self.buffer, batch_size)   # 예: 아무 데서나 64줄

        # 뽑아온 것은 (상황, 행동, 점수, 다음상황, 끝났나) 묶음들의 목록입니다.
        # zip(*batch) 는 이걸 세로로 갈라 줍니다 —
        #   상황은 상황끼리, 행동은 행동끼리 따로 모아 줍니다.
        s, a, r, s_next, done = zip(*batch)

        # 파이썬 목록 -> 넘파이 배열 -> 파이토치 텐서 순서로 바꿉니다.
        # 왜 np.array 를 한 번 거칠까요?
        #   배열들의 '목록'을 텐서로 바로 만들면 파이토치가 하나씩 복사하느라
        #   아주 느려집니다. 넘파이로 먼저 한 덩어리를 만들면 훨씬 빠릅니다.
        return (
            torch.as_tensor(np.array(s), dtype=torch.float32),       # 상황들
            torch.as_tensor(np.array(a), dtype=self.action_dtype),   # 행동들
            torch.as_tensor(np.array(r), dtype=torch.float32),       # 점수들
            torch.as_tensor(np.array(s_next), dtype=torch.float32),  # 다음 상황들
            torch.as_tensor(np.array(done), dtype=torch.float32),    # 끝났나 (1이면 끝)
        )

    def __len__(self):
        # len(buffer) 라고 쓰면 이 함수가 불립니다 — 지금 몇 줄 적혀 있는지 알려 줍니다
        return len(self.buffer)

### 3 / 6 칸

In [ ]:
# ── 오늘 이 교시 — DDPG 소개 ──
import torch                                   # 파이토치
import torch.nn as nn                           # 신경망 부품 상자

### 4 / 6 칸

In [ ]:
# ============================================================
# 오늘부터 행동이 달라집니다.
#   어제까지 : "왼쪽? 오른쪽?" — 고르는 문제 (이산)
#   오늘부터 : "힘을 1.37만큼" — 값을 정하는 문제 (연속)
#
# 왜 어제 방식이 안 통할까요?
#   어제는 모든 행동의 Q값을 내놓고 그중 max 를 골랐습니다.
#   그런데 힘의 크기는 -2.000 부터 2.000 까지 무한히 많습니다.
#   전부 계산해서 max 를 고르는 게 불가능합니다.
#   -> 그래서 "행동을 직접 내놓는 신경망"을 따로 둡니다. 그게 Actor 입니다.
# ============================================================


class Actor(nn.Module):
    """
    상황을 받아서 '할 행동'을 바로 내놓는 신경망.

    어제 정책망과 다른 점:
      어제는 확률을 내놓고 뽑았습니다 (0번 70%, 1번 30%)
      오늘은 값 하나를 딱 정합니다 (힘 1.37) — 이걸 '결정적'이라고 합니다.
    """

    def __init__(self, state_dim, action_dim, max_action):
        # state_dim  = 상황을 나타내는 숫자 개수 (Pendulum 은 3개)
        # action_dim = 행동 값이 몇 개인지 (Pendulum 은 1개 — 회전시킬 힘)
        # max_action = 행동의 최대 크기 (Pendulum 은 2.0)
        super().__init__()                      # 부모 준비 — 빠뜨리면 오류

        self.net = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),      # 3개 -> 256개, 구부리기
            nn.Linear(256, 256), nn.ReLU(),            # 256 -> 256, 또 구부리기
            nn.Linear(256, action_dim), nn.Tanh(),     # 256 -> 1개, 그리고 Tanh
        )
        # 왜 마지막에 Tanh 를 붙일까요?
        #   Tanh 는 어떤 숫자가 들어와도 -1 ~ +1 사이로 눌러 줍니다.
        #   그래야 행동이 엉뚱하게 큰 값(예: 500)이 되는 것을 막습니다.
        #   어제 Q값에는 활성화를 안 붙였는데, 그건 값의 범위가 정해져
        #   있지 않았기 때문입니다. 행동은 범위가 정해져 있습니다.

        self.max_action = max_action            # 나중에 곱해 줄 배율을 기억해 둔다

    def forward(self, s):
        # Tanh 로 -1~1 이 된 값에 max_action 을 곱해 실제 범위로 늘린다
        #   예: 0.685 x 2.0 = 1.37
        return self.net(s) * self.max_action

### 5 / 6 칸

In [ ]:
class Critic(nn.Module):
    """
    (상황, 행동) 을 함께 받아서 "이 조합이 얼마나 좋은지" 점수를 매기는 신경망.

    ★ 어제 DQN 과 가장 크게 다른 점 ★
      어제 : 상황만 받고 -> 모든 행동의 Q값을 한꺼번에 내놓음
      오늘 : 상황 + 행동을 같이 받고 -> 그 하나에 대한 Q값만 내놓음
      행동이 무한히 많으니 "다 내놓기"가 불가능하기 때문입니다.
    """

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256), nn.ReLU(),
            #          ^^^^^^^^^^^^^^^^^^^^^ 상황과 행동을 붙여서 넣으므로 크기를 더한다
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1),                  # 결과는 점수 하나
        )

    def forward(self, s, a):
        # torch.cat = 두 텐서를 옆으로 이어 붙이기
        #   상황 3개 + 행동 1개 = 4개짜리 한 줄이 됩니다.
        # dim=-1 = "맨 마지막 축 방향으로" 붙인다는 뜻
        # squeeze(-1) = (배치, 1) -> (배치,) 로 눌러 모양 맞추기
        return self.net(torch.cat([s, a], dim=-1)).squeeze(-1)

### 6 / 6 칸

In [ ]:
def soft_update(target, source, tau=0.005):
    """
    과녁(타깃 네트워크)을 조금씩만 따라오게 하는 함수. DDPG 와 SAC 가 함께 씁니다.

    어제와 무엇이 다른가요?
      어제 : 20판마다 과녁을 통째로 갈아 끼웠습니다 (계단처럼 툭 바뀜)
      오늘 : 매번 0.5% 씩만 섞습니다 (미끄러지듯 천천히 따라옴)

    왜 바꾸나요?
      연속 행동은 값이 아주 예민합니다. 과녁이 계단처럼 툭툭 튀면
      학습이 그때마다 흔들리다 무너집니다. 그래서 천천히 섞습니다.
    """
    # zip = 두 신경망의 숫자 뭉치를 짝지어 하나씩 꺼낸다
    for tp, sp in zip(target.parameters(), source.parameters()):
        # 새 과녁 = 0.5% 는 최신 것 + 99.5% 는 원래 과녁
        tp.data.copy_(tau * sp.data + (1 - tau) * tp.data)
        # .data 를 쓰는 이유: 이건 '학습'이 아니라 '복사'입니다.
        #   미분 기록을 남기지 않고 값만 바꿔치기합니다.

---

## 막히면

- 사이트의 같은 교시를 보세요 — 실행 결과와 해설이 그대로 있습니다.
  https://pytorch26.dreamitbiz.com/#/day/3/1
- 오류가 나면 **[막힐 때]** 메뉴부터.
  https://pytorch26.dreamitbiz.com/#/help

---

*Ph.D Aebon & Claude Code 협작 전자출판 도서 · © 2026 DreamIT Biz*